In [6]:
import os
import pywt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools as it
import datetime
import pickle
import os

In [5]:
# !pip install PyWavelets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 31.1 MB/s eta 0:00:00


In [7]:
# Specify the filename for the pickled file
#dataset_3s_path = "/mnt/d/Personal/Science/Smart Watch/motionsensors-ml/datasets/dataframes_3s_20240408_095235.csv"
#dataset_3s_path = '/Users/olena/myProjects/bitbucket/agv_ann/Sląska Politechnika/SmartIndustry_lab/motionsensorsML 2/datasets/dataset_20240413_141813/dataframes_3s_20240413_141813.csv'
#output_path = "/mnt/d/Personal/Science/Smart Watch/motionsensors-ml/datasets/"
#output_path = '/Users/olena/myProjects/bitbucket/agv_ann/Sląska Politechnika/SmartIndustry_lab/motionsensorsML 2/datasets/'

from google.colab import drive
drive.mount('/content/drive')

# dataset_3s_path =  '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240413_141813/dataframes_3s_20240413_141813.csv'
# output_path = '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240413_141813/'

#dataset_3s_path =  '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240429_193334/dataframes_3s_20240429_193334.csv'
#output_path = '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240429_193334/'

#dataset_3s_path =  '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240430_195036/dataframes_3s_20240430_195036.csv'
#output_path = '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240430_195036/'

#dataset_3s_path =  '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/dataframes_3s_20240526_175526.csv'
#output_path = '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/'


dataset_3s_path =  '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/dataframes_3s_20240526_175526.csv'
output_path = '/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/'

dataset = pd.read_csv(dataset_3s_path, header=None)

print("Data loaded from the csv file.")

Mounted at /content/drive
Data loaded from the csv file.


In [8]:
#
# Constants defining the structure of the CSV file
#

# File "aggregates*.csv"
AGGREGATES_ROW_REGISTERED_TIMESTAMP = 0
AGGREGATES_ROW_AGGREGATE_ID         = 1
AGGREGATES_ROW_AGGREGATE_EVENT_ID	= 2
AGGREGATES_ROW_MAP_ID               = 3
AGGREGATES_ROW_SUBJECT_ID           = 4

# File "dataframes_3s_*.csv"
DATAFRAMES_3S_ROW_START_TIMESTAMP         = 0
DATAFRAMES_3S_ROW_END_TIMESTAMP           = 1
DATAFRAMES_3S_ROW_ACTIVITY_ID	          = 2
DATAFRAMES_3S_ROW_DEVICE_BODY_LOCATION_ID = 3
DATAFRAMES_3S_ROW_SUBJECT_ID              = 4
DATAFRAMES_3S_ROW_ACCELEROMETER_X_START	  = 5
DATAFRAMES_3S_ROW_ACCELEROMETER_X_END     = 304
DATAFRAMES_3S_ROW_ACCELEROMETER_Y_START	  = 305
DATAFRAMES_3S_ROW_ACCELEROMETER_Y_END     = 604
DATAFRAMES_3S_ROW_ACCELEROMETER_Z_START   = 605
DATAFRAMES_3S_ROW_ACCELEROMETER_Z_END     = 904
DATAFRAMES_3S_ROW_GYROSCOPE_X_START       = 905
DATAFRAMES_3S_ROW_GYROSCOPE_X_END         = 1204
DATAFRAMES_3S_ROW_GYROSCOPE_Y_START       = 1205
DATAFRAMES_3S_ROW_GYROSCOPE_Y_END         = 1504
DATAFRAMES_3S_ROW_GYROSCOPE_Z_START       = 1505
DATAFRAMES_3S_ROW_GYROSCOPE_Z_END         = 1804

In [9]:
def mapImage(im):
    # Get the min and max of coefAx
    min_val = np.min(im)
    max_val = np.max(im)

    # Map coefAx to [0, 1] range
    mapped_im = (im - min_val) / (max_val - min_val)

    return mapped_im[::-1].astype(np.float32)

In [10]:
scales = np.arange(1, 257)
mother = "morl"

In [11]:
s_frames = []
s_labels = []

for index, dataframe in dataset.iterrows():
    AccelX = dataframe.iloc[DATAFRAMES_3S_ROW_ACCELEROMETER_X_START:DATAFRAMES_3S_ROW_ACCELEROMETER_X_END + 1]
    AccelY = dataframe.iloc[DATAFRAMES_3S_ROW_ACCELEROMETER_Y_START:DATAFRAMES_3S_ROW_ACCELEROMETER_Y_END + 1]
    AccelZ = dataframe.iloc[DATAFRAMES_3S_ROW_ACCELEROMETER_Z_START:DATAFRAMES_3S_ROW_ACCELEROMETER_Z_END + 1]
    GyroX =  dataframe.iloc[DATAFRAMES_3S_ROW_GYROSCOPE_X_START:DATAFRAMES_3S_ROW_GYROSCOPE_X_END + 1]
    GyroY =  dataframe.iloc[DATAFRAMES_3S_ROW_GYROSCOPE_Y_START:DATAFRAMES_3S_ROW_GYROSCOPE_Y_END + 1]
    GyroZ =  dataframe.iloc[DATAFRAMES_3S_ROW_GYROSCOPE_Z_START:DATAFRAMES_3S_ROW_GYROSCOPE_Z_END + 1]
    ClassID = dataframe.iloc[DATAFRAMES_3S_ROW_ACTIVITY_ID]

    coefAx, freqsAx = pywt.cwt(AccelX, scales, mother)
    coefAy, freqsAy = pywt.cwt(AccelY, scales, mother)
    coefAz, freqsAz = pywt.cwt(AccelZ, scales, mother)
    coefGx, freqsGx = pywt.cwt(GyroX, scales, mother)
    coefGy, freqsGy = pywt.cwt(GyroY, scales, mother)
    coefGz, freqsGz = pywt.cwt(GyroZ, scales, mother)

    # Save as a file
    frame = np.dstack((
        mapImage(abs(coefAx)),
        mapImage(abs(coefAy)),
        mapImage(abs(coefAz)),
        mapImage(abs(coefGx)),
        mapImage(abs(coefGy)),
        mapImage(abs(coefGz)),
    ))

    s_frames.append(frame)
    s_labels.append(ClassID)

In [12]:
# Get the current timestamp and format it for the filename
current_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Specify the filename for the pickled file
pickle_filename = f"{output_path}dataset_np.pickle"

# Open a file in binary write mode and pickle the object
with open(pickle_filename, "wb") as file:
    pickle.dump(s_frames, file)

print(f"'{pickle_filename}' has been created with the pickled data.")

'/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/dataset_np.pickle' has been created with the pickled data.


In [13]:
# Get the current timestamp and format it for the filename
current_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Specify the filename for the pickled file
pickle_filename = f"{output_path}lables_np.pickle"

# Open a file in binary write mode and pickle the object
with open(pickle_filename, "wb") as file:
    pickle.dump(s_labels, file)

print(f"'{pickle_filename}' has been created with the pickled data.")

'/content/drive/MyDrive/Pol_Sl/SmartIndustry_Lab/dataset_20240526_175526/lables_np.pickle' has been created with the pickled data.
